# Nature Dependency: Same Sector, Different Country — Different Risk
## Greenings Portfolio · ENCORE + WWF Risk Filter

**Core argument:** ENCORE ecosystem-service dependencies are **sector-determined** — two hospitals in different countries have identical base dependency scores.  
But the **WWF Risk Filter** is geography-specific: water scarcity, biodiversity threat, and regulatory pressure differ sharply by country.  
The product of both — **effective dependency risk** — reveals that nominally similar projects carry fundamentally different nature-related exposure depending on where they operate.

| Section | What it shows |
|---------|---------------|
| 1 | ENCORE dependency heatmap by sector group |
| 2 | Country-specific WWF risk (physical + biodiversity) |
| 3 | Effective risk heatmap: ENCORE × country-risk |
| 4 | Radar: within-sector country comparison (Hospital & Hydro) |
| 5 | Portfolio bubble: investment × country risk |
| 6 | Revenue-at-risk by project and sector |

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display

warnings.filterwarnings("ignore")

ROOT    = Path("..").resolve()
DEP_DIR = ROOT / "tvp_dependency" / "assessment_output"
FIG_DIR = ROOT / "project_assessment" / "greenings_backtesting"

print("ROOT   :", ROOT)
print("DEP_DIR:", DEP_DIR)

In [ ]:
# ── Project metadata ──────────────────────────────────────────────────────────
PROJECT_META = {
    "Proj_001":    dict(sector="Hospital",  country="LATAM (Brazil)",    region="latam",   invest_m=250.0,  display="HOSP-1"),
    "Proj_002":    dict(sector="Hospital",  country="Africa (Nigeria)",  region="africa",  invest_m=25.0,   display="HOSP-2"),
    "Proj_003":    dict(sector="Hospital",  country="Europe (Germany)",  region="europe",  invest_m=75.0,   display="HOSP-3"),
    "Hydro_AF":    dict(sector="Hydro",     country="Africa (Nigeria)",  region="africa",  invest_m=30.0,   display="HYDRO-AF"),
    "Hydro_AS":    dict(sector="Hydro",     country="Asia (Malaysia)",   region="asia",    invest_m=150.0,  display="HYDRO-AS"),
    "Hydro_EU":    dict(sector="Hydro",     country="Europe (Germany)",  region="europe",  invest_m=2.0,    display="HYDRO-EU"),
    "Rail_EU_DEV": dict(sector="Rail",      country="Europe (Germany)",  region="europe",  invest_m=1998.0, display="RAIL-DEV"),
    "Rail_EU_OP1": dict(sector="Rail",      country="Europe (Germany)",  region="europe",  invest_m=0.14,   display="RAIL-OP1"),
    "Rail_EU_OP2": dict(sector="Rail",      country="Europe (Germany)",  region="europe",  invest_m=0.10,   display="RAIL-OP2"),
}
PROJECT_ORDER = ["Proj_001","Proj_002","Proj_003",
                 "Hydro_AF","Hydro_AS","Hydro_EU",
                 "Rail_EU_DEV","Rail_EU_OP1","Rail_EU_OP2"]

SECTOR_COLORS = {"Hospital": "#3498db", "Hydro": "#1abc9c", "Rail": "#e67e22"}
COUNTRY_COLORS = {
    "LATAM (Brazil)":   "#f39c12",
    "Africa (Nigeria)": "#e74c3c",
    "Asia (Malaysia)":  "#9b59b6",
    "Europe (Germany)": "#2ecc71",
}

# ── Load ENCORE dependency files ──────────────────────────────────────────────
enc_frames = []
for f in sorted(DEP_DIR.glob("encore_materiality_*.csv")):
    pid = f.stem.replace("encore_materiality_", "")
    df  = pd.read_csv(f)
    df["project_id"] = pid
    enc_frames.append(df)
enc = pd.concat(enc_frames, ignore_index=True)

for pid, meta in PROJECT_META.items():
    enc.loc[enc.project_id == pid, "sector"]  = meta["sector"]
    enc.loc[enc.project_id == pid, "country"] = meta["country"]
    enc.loc[enc.project_id == pid, "display"] = meta["display"]

# ── Load WWF risk scores ───────────────────────────────────────────────────────
wwf = pd.read_csv(DEP_DIR / "wwf_risk_scores.csv")
for pid, meta in PROJECT_META.items():
    wwf.loc[wwf.project_id == pid, "sector"]  = meta["sector"]
    wwf.loc[wwf.project_id == pid, "country"] = meta["country"]
    wwf.loc[wwf.project_id == pid, "display"] = meta["display"]
    wwf.loc[wwf.project_id == pid, "invest_m"] = meta["invest_m"]

# ── Load stress test (revenue at risk) ────────────────────────────────────────
stress = pd.read_csv(DEP_DIR / "portfolio_stress_test.csv")
for pid, meta in PROJECT_META.items():
    stress.loc[stress.project_id == pid, "sector"]  = meta["sector"]
    stress.loc[stress.project_id == pid, "country"] = meta["country"]
    stress.loc[stress.project_id == pid, "display"] = meta["display"]

print(f"ENCORE records loaded: {len(enc)}")
print(f"WWF records: {len(wwf)}")

## Section 1 · ENCORE Dependency Heatmap by Sector

ENCORE assigns ecosystem-service dependency scores at the **sector level**.  
Within each sector group the three country variants carry identical base scores — there is no geographic differentiation here.

In [ ]:
# ── Build pivot: rows = services, cols = projects (ordered by sector then country)
pivot_enc = (
    enc
    .pivot_table(index="ecosystem_service", columns="project_id",
                 values="dependency_score", aggfunc="first")
    .reindex(columns=PROJECT_ORDER)
)
# Keep only rows where at least one project scores > 0
pivot_enc = pivot_enc[pivot_enc.max(axis=1) > 0].sort_values(PROJECT_ORDER[0], ascending=False)

col_labels  = [PROJECT_META[p]["display"] + "\n" + PROJECT_META[p]["country"].split(" ")[0]
               for p in PROJECT_ORDER]
col_sectors = [PROJECT_META[p]["sector"] for p in PROJECT_ORDER]

cmap_dep = LinearSegmentedColormap.from_list("dep", ["#f7fbff","#c6dbef","#6baed6","#2171b5","#08306b"])

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(pivot_enc.values, cmap=cmap_dep, vmin=0, vmax=5, aspect="auto")

ax.set_xticks(range(len(PROJECT_ORDER)))
ax.set_xticklabels(col_labels, fontsize=8)
ax.set_yticks(range(len(pivot_enc)))
ax.set_yticklabels([s.replace("_"," ") for s in pivot_enc.index], fontsize=8)

# Score labels inside cells
for i, svc in enumerate(pivot_enc.index):
    for j, pid in enumerate(PROJECT_ORDER):
        val = pivot_enc.loc[svc, pid]
        if val > 0:
            ax.text(j, i, str(int(val)), ha="center", va="center",
                    fontsize=8, color="white" if val >= 3 else "#333")

# Sector bracket labels on top
sector_spans = [(0,2,"Hospital"),(3,5,"Hydro"),(6,8,"Rail")]
for s, e, label in sector_spans:
    mid = (s + e) / 2
    ax.annotate("", xy=(e+0.4, -0.9), xytext=(s-0.4, -0.9), xycoords="data",
                arrowprops=dict(arrowstyle="-", color=SECTOR_COLORS[label], lw=2))
    ax.text(mid, -1.4, label, ha="center", va="top", fontsize=9,
            fontweight="bold", color=SECTOR_COLORS[label],
            transform=ax.transData)

# Vertical dividers between sector groups
for x in [2.5, 5.5]:
    ax.axvline(x, color="white", lw=2)

plt.colorbar(im, ax=ax, shrink=0.6, label="Dependency score (0=none → 5=very high)")
ax.set_title(
    "ENCORE Ecosystem Service Dependency Scores\n"
    "(sector-determined: identical within each sector group regardless of country)",
    fontsize=11, fontweight="bold", pad=20
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_dep_encore_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()

## Section 2 · Country-Specific Risk: WWF Water & Biodiversity Filters

The WWF Risk Filter scores physical water risk (WRF) and biodiversity risk (BRF) at the project location.  
These scores are **country-specific** and vary substantially even within the same sector.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle(
    "WWF Risk Filter Scores by Country · Same Sector, Different Geography",
    fontsize=12, fontweight="bold"
)

for ax, sector in zip(axes, ["Hospital", "Hydro", "Rail"]):
    subset = wwf[wwf.sector == sector].copy()
    subset = subset.drop_duplicates("project_id")
    # For Rail all Europe — use display names to differentiate
    subset["label"] = subset["display"] + "\n" + subset["country"].str.split(" ").str[0]

    x = np.arange(len(subset))
    w = 0.35
    colors = [COUNTRY_COLORS.get(c, "#95a5a6") for c in subset["country"]]

    b1 = ax.bar(x - w/2, subset["wrf_composite"], width=w, label="WRF (physical water)",
                color=colors, alpha=0.85, edgecolor="white")
    b2 = ax.bar(x + w/2, subset["brf_composite"], width=w, label="BRF (biodiversity)",
                color=colors, alpha=0.5,  edgecolor="white", hatch="//")

    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7.5)

    # High-risk threshold line
    ax.axhline(3.5, color="#c0392b", lw=1, ls="--", alpha=0.7)
    ax.text(len(subset)-0.5, 3.55, "high-risk\nthreshold", ha="right", fontsize=6.5, color="#c0392b")

    ax.set_xticks(x)
    ax.set_xticklabels(subset["label"], fontsize=8)
    ax.set_ylim(0, 5.2)
    ax.set_title(sector, fontsize=10, fontweight="bold",
                 color=SECTOR_COLORS[sector], pad=6)
    ax.spines[["top","right"]].set_visible(False)
    if ax == axes[0]:
        ax.set_ylabel("Risk score (0–5)", fontsize=9)

# Legend
solid_patch  = mpatches.Patch(color="#888", alpha=0.85, label="WRF — physical water risk")
hatch_patch  = mpatches.Patch(color="#888", alpha=0.5,  hatch="//", label="BRF — biodiversity risk")
country_patches = [mpatches.Patch(color=c, label=l) for l, c in COUNTRY_COLORS.items()]
fig.legend(
    handles=[solid_patch, hatch_patch] + country_patches,
    loc="lower center", ncol=6, fontsize=8, framealpha=0.9,
    bbox_to_anchor=(0.5, -0.08)
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_dep_wwf_country.png", dpi=130, bbox_inches="tight")
plt.show()

## Section 3 · Effective Dependency Risk = ENCORE × Country Risk

**Effective risk** = `dependency_score × (wrf_composite / 5)`.  
This multiplies the sector's structural exposure to each ecosystem service by the country-specific pressure on that service.  
The result breaks the within-sector uniformity seen in Section 1 and makes country a primary differentiator.

In [ ]:
# ── Build effective risk pivot ────────────────────────────────────────────────
wrf_map = dict(zip(wwf.project_id, wwf.wrf_composite))

enc["effective_risk"] = enc["dependency_score"] * enc["project_id"].map(wrf_map) / 5.0

pivot_eff = (
    enc
    .pivot_table(index="ecosystem_service", columns="project_id",
                 values="effective_risk", aggfunc="first")
    .reindex(columns=PROJECT_ORDER)
)
# Keep same rows as the base heatmap
pivot_eff = pivot_eff.reindex(pivot_enc.index)

# ── Side-by-side: base vs effective ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

cmap_base = LinearSegmentedColormap.from_list("base", ["#f7fbff","#c6dbef","#6baed6","#2171b5","#08306b"])
cmap_eff  = LinearSegmentedColormap.from_list("eff",  ["#fff5f0","#fcbba1","#fb6a4a","#cb181d","#67000d"])

for ax, pivot, cmap, title, vmax, label in [
    (ax1, pivot_enc, cmap_base, "Base ENCORE dependency score\n(sector-determined — identical within sector group)", 5, "Dependency (0–5)"),
    (ax2, pivot_eff, cmap_eff,  "Effective risk = ENCORE × country WRF / 5\n(country-adjusted — diverges within sector group)",  5, "Effective risk (0–5)"),
]:
    im = ax.imshow(pivot.values.astype(float), cmap=cmap, vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(PROJECT_ORDER)))
    ax.set_xticklabels(col_labels, fontsize=7.5)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels([s.replace("_", " ") for s in pivot.index], fontsize=8)

    for i, svc in enumerate(pivot.index):
        for j, pid in enumerate(PROJECT_ORDER):
            val = pivot.iloc[i, j]
            if pd.notna(val) and val > 0:
                ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                        fontsize=7.5, color="white" if val >= 3 else "#333")

    for x in [2.5, 5.5]:
        ax.axvline(x, color="white", lw=2.5)
    for s, e, lbl in [(0,2,"Hospital"),(3,5,"Hydro"),(6,8,"Rail")]:
        ax.text((s+e)/2, -1.3, lbl, ha="center", va="top", fontsize=9,
                fontweight="bold", color=SECTOR_COLORS[lbl], transform=ax.transData)

    plt.colorbar(im, ax=ax, shrink=0.55, label=label)
    ax.set_title(title, fontsize=10, fontweight="bold", pad=18)

plt.suptitle(
    "Country Location Breaks Within-Sector Uniformity",
    fontsize=13, fontweight="bold", y=1.03
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_dep_effective_risk_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()

## Section 4 · Radar: Within-Sector Country Comparison

Radar charts for **Hospital** and **Hydro** — the two sectors with genuine geographic spread.  
Each spoke is a material ecosystem service; each line is a country variant.  
The divergence between lines within the same chart demonstrates the country effect.

In [ ]:
def radar_chart(ax, categories, values_by_group, colors, title):
    """Draw a radar chart on ax. values_by_group: {label: [v1..vN]}."""
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close polygon

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([c.replace("_"," ") for c in categories], size=7.5)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1","2","3","4","5"], size=6, color="#999")
    ax.spines["polar"].set_color("#ddd")
    ax.grid(color="#eee", lw=0.8)

    for (label, vals), color in zip(values_by_group.items(), colors):
        vals_closed = vals + vals[:1]
        ax.plot(angles, vals_closed, color=color, lw=2, label=label)
        ax.fill(angles, vals_closed, color=color, alpha=0.10)

    ax.set_title(title, size=11, fontweight="bold", pad=16)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)


fig = plt.figure(figsize=(16, 7))

# ── Hospital radar ────────────────────────────────────────────────────────────
hosp_svcs = ["water_supply","air_quality_regulation","climate_regulation","genetic_materials"]
hosp_projects = {
    "HOSP · LATAM (Brazil)":    "Proj_001",
    "HOSP · Africa (Nigeria)":  "Proj_002",
    "HOSP · Europe (Germany)": "Proj_003",
}
hosp_colors = [COUNTRY_COLORS["LATAM (Brazil)"], COUNTRY_COLORS["Africa (Nigeria)"],
               COUNTRY_COLORS["Europe (Germany)"]]

ax1 = fig.add_subplot(121, polar=True)
hosp_vals = {}
for label, pid in hosp_projects.items():
    eff = enc[enc.project_id == pid].set_index("ecosystem_service")["effective_risk"]
    hosp_vals[label] = [float(eff.get(s, 0)) for s in hosp_svcs]
radar_chart(ax1, hosp_svcs, hosp_vals, hosp_colors,
            "Hospital\n(3 countries, same ENCORE sector)")

# ── Hydro radar ───────────────────────────────────────────────────────────────
hydro_svcs = ["water_supply","water_flow_regulation","freshwater_ecosystem_use",
              "climate_regulation","flood_and_storm_protection","erosion_control"]
hydro_projects = {
    "HYDRO · Africa (Nigeria)":  "Hydro_AF",
    "HYDRO · Asia (Malaysia)":   "Hydro_AS",
    "HYDRO · Europe (Germany)": "Hydro_EU",
}
hydro_colors = [COUNTRY_COLORS["Africa (Nigeria)"], COUNTRY_COLORS["Asia (Malaysia)"],
                COUNTRY_COLORS["Europe (Germany)"]]

ax2 = fig.add_subplot(122, polar=True)
hydro_vals = {}
for label, pid in hydro_projects.items():
    eff = enc[enc.project_id == pid].set_index("ecosystem_service")["effective_risk"]
    hydro_vals[label] = [float(eff.get(s, 0)) for s in hydro_svcs]
radar_chart(ax2, hydro_svcs, hydro_vals, hydro_colors,
            "Hydro Power\n(3 continents, same ENCORE sector)")

fig.suptitle(
    "Effective Dependency Risk by Country · Hospital and Hydro Sectors\n"
    "(divergence within each chart = country effect on identical base dependencies)",
    fontsize=12, fontweight="bold", y=1.04
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_dep_radar_country.png", dpi=130, bbox_inches="tight")
plt.show()

## Section 5 · Portfolio Bubble: Investment vs Country Risk

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for _, row in wwf.iterrows():
    pid    = row["project_id"]
    sector = PROJECT_META[pid]["sector"]
    color  = SECTOR_COLORS[sector]
    size   = max(40, PROJECT_META[pid]["invest_m"] * 0.18)

    ax.scatter(row["wrf_composite"], row["brf_composite"],
               s=size, color=color, alpha=0.75, edgecolors="white", linewidth=1.2)
    ax.annotate(
        PROJECT_META[pid]["display"] + "\n" + PROJECT_META[pid]["country"].split(" ")[0],
        xy=(row["wrf_composite"], row["brf_composite"]),
        xytext=(6, 4), textcoords="offset points",
        fontsize=7.5, color="#333"
    )

# Quadrant lines at 3.5
ax.axhline(3.5, color="#c0392b", lw=1, ls="--", alpha=0.5)
ax.axvline(3.5, color="#c0392b", lw=1, ls="--", alpha=0.5)
ax.text(3.52, ax.get_ylim()[0] + 0.05, "high physical risk →", fontsize=7, color="#c0392b")
ax.text(ax.get_xlim()[0] + 0.02, 3.52, "high bio risk ↑", fontsize=7, color="#c0392b")

# Legend: sectors
for sector, color in SECTOR_COLORS.items():
    ax.scatter([], [], s=80, color=color, label=sector, alpha=0.75)
ax.scatter([], [], s=20,  color="#888", alpha=0.5, label="small investment")
ax.scatter([], [], s=200, color="#888", alpha=0.5, label="large investment")
ax.legend(loc="lower right", fontsize=8, framealpha=0.9)

ax.set_xlabel("WRF composite — physical water risk (0–5)", fontsize=10)
ax.set_ylabel("BRF composite — biodiversity risk (0–5)", fontsize=10)
ax.set_title(
    "Portfolio: Physical Water Risk vs Biodiversity Risk\n"
    "(bubble size ∝ investment MUSD · dashed line = high-risk threshold 3.5)",
    fontsize=11, fontweight="bold"
)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_dep_portfolio_bubble.png", dpi=130, bbox_inches="tight")
plt.show()

## Section 6 · Revenue at Risk by Project and Sector

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle(
    "Revenue at Risk (USD M) by Country · Same Sector, Different Exposure",
    fontsize=12, fontweight="bold"
)

for ax, sector in zip(axes, ["Hospital", "Hydro", "Rail"]):
    sub = stress[stress.sector == sector].copy()
    sub["label"] = sub["display"] + "\n" + sub["country"].str.split(" ").str[0]
    colors = [COUNTRY_COLORS.get(c, "#95a5a6") for c in sub["country"]]

    bars = ax.bar(sub["label"], sub["revenue_at_risk_usd_m"],
                  color=colors, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, sub["revenue_at_risk_usd_m"]):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, val + sub["revenue_at_risk_usd_m"].max()*0.02,
                    f"${val:.1f}M", ha="center", va="bottom", fontsize=8, fontweight="bold")
        else:
            ax.text(bar.get_x() + bar.get_width()/2, 0.5,
                    "$0", ha="center", va="bottom", fontsize=8, color="#aaa")

    ax.set_title(sector, fontsize=10, fontweight="bold",
                 color=SECTOR_COLORS[sector], pad=6)
    ax.set_ylabel("Revenue at risk (USD M)", fontsize=8)
    ax.spines[["top","right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=8)

fig.legend(
    handles=[mpatches.Patch(color=c, label=l) for l, c in COUNTRY_COLORS.items()],
    loc="lower center", ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.08)
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_dep_revenue_at_risk.png", dpi=130, bbox_inches="tight")
plt.show()

## Summary Table

In [ ]:
summary = stress[[
    "display", "sector", "country", "investment_usd_m",
    "encore_n_material", "top_dependency",
    "wrf_composite", "brf_composite",
    "overall_high_risk", "revenue_at_risk_usd_m"
]].copy()
summary.columns = [
    "Project", "Sector", "Country", "CAPEX (MUSD)",
    "# Material services", "Top dependency",
    "WRF (water)", "BRF (biodiversity)",
    "High risk", "Revenue at risk (MUSD)"
]
summary = summary.set_index(["Sector", "Project"]).sort_index()

display(
    summary.style
    .format({
        "CAPEX (MUSD)":          "{:,.1f}",
        "WRF (water)":           "{:.3f}",
        "BRF (biodiversity)":    "{:.3f}",
        "Revenue at risk (MUSD)": "{:.2f}",
    })
    .background_gradient(subset=["WRF (water)","BRF (biodiversity)"],
                         cmap="YlOrRd", vmin=2.5, vmax=4.5)
    .background_gradient(subset=["Revenue at risk (MUSD)"],
                         cmap="Reds", vmin=0)
    .set_caption(
        "Nature-related dependency comparison · "
        "identical ENCORE scores within sector, country-driven risk divergence"
    )
)

## Methodology Notes

| Item | Detail |
|------|--------|
| **ENCORE dependency** | Sector-level rating (1–5) of how much each business activity depends on a given ecosystem service. Ratings are identical for all projects in the same sector archetype. Source: ENCORE tool v2.0, 2023. |
| **WWF WRF** | Water Risk Filter composite (physical sub-score: scarcity, variability, flood/drought; regulatory sub-score; reputational sub-score). Country + sector specific. |
| **WWF BRF** | Biodiversity Risk Filter composite (species threat, ecosystem quality, protected-area overlap). Country specific. |
| **Effective risk** | `dependency_score × WRF_composite / 5` — scales the structural dependency by the country's physical water stress. A Hospital in LATAM and a Hospital in Germany have identical ENCORE scores but different effective risk because Brazil faces greater water scarcity. |
| **Revenue at risk** | Estimated annual revenue exposure to nature disruption: `revenue_proxy × P(disruption)` where P is derived from overall high-risk flag and WRF/BRF composite scores. |
| **Rail note** | All three Rail projects are located in Germany — country variation within the Rail sector is absent in this portfolio. Variation between Rail projects reflects investment stage (development vs operational). |